# Прогноз выполнения плана взысканий

Самостоятельный ноутбук для любого целевого месяца. Имя файла сохранено для совместимости.
Все определения находятся в ячейках; пакет проекта для запуска не нужен.

Прогноз фиксируется **до начала месяца**, использует известный план по датам и проверяется по разнице
`100 * abs(прогноз_накопленный - факт_накопленный) / план_накопленный` в процентных пунктах.
Цель — не больше 1 п.п. на каждой дате; достижение цели на рабочих данных пока не подтверждено.

Перед запуском выберите смысл входных сумм: `daily` — за день, `cumulative` — с начала месяца/входа когорты.
Режимы факта и плана независимы. Накопленные снимки должны быть полными, с устойчивыми ключами когорт.
Пропуски не заменяются нулями, отрицательные корректировки сохраняются.

Установите зависимости в используемое ядро, если нужно:
```python
%pip install "numpy>=1.26,<3" "pandas>=2.2,<4" "plotly>=5.20,<8" "matplotlib>=3.8,<4" "catboost>=1.2,<2" "statsmodels>=0.14,<0.16" "holidays>=0.60,<1"
```

**Простой вход**: pandas DataFrame `df` с `date, fact, plan` за историю и целевой месяц;
факт целевого месяца может отсутствовать (`NaN`). Даты не подменяются. Дополнительные сегментные колонки сохраняются.
**Детальный вход**: `run(history_df, plan_df, actual_df, config=config)` — история RR, планы всех месяцев,
отдельный необязательный факт целевого месяца. Ключи: `report_month, report_dt, dt_in, subproduct, dpd_bucket,
pvr_bucket, terbank, inflow`. История содержит `fact` или `w1,w2,cession,write_off` и `target`;
план содержит `target`. Для исходных RR-компонентов нужны также `acc_count,avg_dpd,avg_pvr`.

Если заданы `plan_available_column`/`raw_available_column`, используются только доступные на момент прогноза
версии. Без них предполагается, что пользователь передал исторически доступные планы и снимки;
поздние пересмотры из финальной выгрузки восстановить невозможно.

Первые месяцы используются для обучения, следующие — для выбора модели, последние три — для отдельной проверки.
При настройках по умолчанию нужно минимум 21 месяц полной истории. Границы можно задать явно.
Для технической проверки переключите `DEMO = True` ниже. Искусственный пример не подтверждает бизнес-точность.


## config

In [ ]:
from dataclasses import asdict, dataclass, replace
import pandas as pd


@dataclass(frozen=True)
class Config:
    db_path: str = "input/m1p_database_v2.db"
    plan_table: str | None = None  # None means raw_table
    plan_value_column: str = "target"
    plan_is_forecast: int = 0
    plan_available_column: str | None = None
    portfolio_path: str | None = None
    inflow_path: str | None = None
    operations_path: str | None = None
    calendar_path: str | None = None
    reference_path: str | None = None
    output_dir: str = "outputs/forecast"
    forecast_month: str | None = None
    development_start: str | None = None
    development_end: str | None = None
    control_start: str | None = None
    control_end: str | None = None
    fact_mode: str | None = None  # daily or cumulative, explicitly chosen by the caller
    plan_mode: str | None = None
    min_train_months: int = 12
    raw_table: str = "rr1plus_data_raw"
    raw_available_column: str | None = None
    raw_date_encoding: str = "iso"  # iso, yyyymmdd, s, ms, us, ns
    segments: tuple[str, ...] = (
        "subproduct", "dpd_bucket", "pvr_bucket", "terbank", "inflow"
    )
    tolerance_pp: float = 1.0
    seed: int = 42
    reference_months: int = 3
    enable_catboost: bool = True
    enable_sarimax: bool = True
    catboost_iterations: int = 600
    catboost_depths: tuple[int, ...] = (4, 6)
    blend_weights: tuple[float, ...] = (0.0, 0.25, 0.5, 0.75, 1.0)
    country: str = "RU"
    require_all_folds: bool = True
    require_calendar_overrides: bool = False

    def resolved(self):
        if self.forecast_month is None:
            raise ValueError("Specify forecast_month, e.g. Config(forecast_month='2026-11', fact_mode='daily', plan_mode='daily')")
        target = pd.Period(self.forecast_month, "M")
        values = dict(forecast_month=str(target))
        # Six development months followed by three untouched control months.
        for name, offset in [("development_start", 9), ("development_end", 4),
                             ("control_start", 3), ("control_end", 1)]:
            value = getattr(self, name)
            values[name] = str(pd.Period(value, "M")) if value is not None else str(target - offset)
        return replace(self, **values)

    def validate(self):
        if self.fact_mode not in {"daily", "cumulative"} or self.plan_mode not in {"daily", "cumulative"}:
            raise ValueError("Explicitly set fact_mode and plan_mode to 'daily' or 'cumulative' after checking the source report")
        if self.plan_is_forecast not in (0, 1):
            raise ValueError("plan_is_forecast must be 0 or 1; plan rows must not be summed twice")
        if not self.segments or len(set(self.segments)) != len(self.segments):
            raise ValueError("segments must be nonempty and unique")
        if self.min_train_months < 1 or self.tolerance_pp <= 0:
            raise ValueError("Invalid training length or tolerance")
        if self.reference_months < 1 or self.catboost_iterations < 1 or not self.catboost_depths:
            raise ValueError("Reference window and model sizes must be positive")
        if any(not 1 <= d <= 10 for d in self.catboost_depths):
            raise ValueError("catboost_depths must be between 1 and 10")
        if any(not 0 <= weight <= 1 for weight in self.blend_weights):
            raise ValueError("Blend weights must be between zero and one")
        if self.raw_date_encoding not in {"iso", "yyyymmdd", "s", "ms", "us", "ns"}:
            raise ValueError("Unsupported raw_date_encoding")
        if not (pd.Period(self.development_start, 'M') <= pd.Period(self.development_end, 'M')
                < pd.Period(self.control_start, 'M') <= pd.Period(self.control_end, 'M')
                < pd.Period(self.forecast_month, 'M')):
            raise ValueError("Development, control and target periods must be ordered and disjoint")

    def as_dict(self):
        return asdict(self)


def month_start(month):
    return pd.Period(month, freq="M").start_time


def origin(month):
    return month_start(month) - pd.Timedelta(nanoseconds=1)


def days(month):
    p = pd.Period(month, freq="M")
    return pd.date_range(p.start_time, p.end_time.normalize(), freq="D")


## data

In [ ]:
"""Read-only SQLite access and explicit vintage-aware input contracts."""
from pathlib import Path
from contextlib import contextmanager
import hashlib
import re
import sqlite3
import numpy as np
import pandas as pd


def identifier(name):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z_0-9]*", name):
        raise ValueError(f"Unsafe SQL identifier: {name!r}")
    return '"' + name + '"'


def numeric(frame, columns):
    for col in columns:
        frame[col] = pd.to_numeric(frame[col], errors="raise")
        if not np.isfinite(frame[col]).all():
            raise ValueError(f"Missing or non-finite numeric value: {col}")
    return frame


def normalize_segments(frame, segments):
    for col in segments:
        if col not in frame:
            raise ValueError(f"Missing segment column: {col}")
        frame[col] = frame[col].astype("string").fillna("__UNDEFINED__").str.strip()
        frame[col] = frame[col].replace("", "__UNDEFINED__").astype(str)
        if col == "inflow":
            frame[col] = frame[col].replace({"0.0": "0", "1.0": "1"})
    return frame


def parse_dates(values, encoding="iso"):
    if isinstance(values.dtype, pd.DatetimeTZDtype):
        return values.dt.tz_convert("Europe/Moscow").dt.tz_localize(None)
    if pd.api.types.is_datetime64_any_dtype(values.dtype):
        return values
    if encoding == "iso":
        # Naive strings are local Moscow time; offset-aware strings are converted to it.
        parsed = pd.to_datetime(values, format="mixed", errors="raise", utc=True)
        aware = values.astype(str).str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$", regex=True)
        result = parsed.dt.tz_localize(None)
        result.loc[aware] = parsed.loc[aware].dt.tz_convert("Europe/Moscow").dt.tz_localize(None)
        return result
    if encoding == "yyyymmdd":
        return pd.to_datetime(values.astype(str).str.replace(r"\.0$", "", regex=True), format="%Y%m%d")
    return pd.to_datetime(pd.to_numeric(values), unit=encoding, utc=True).dt.tz_convert("Europe/Moscow").dt.tz_localize(None)


def sql_date(column, encoding):
    q = identifier(column)
    if encoding == "iso":
        aware = f"(substr({q},-1)='Z' OR substr({q},-6,1) IN ('+','-') OR substr({q},-5,1) IN ('+','-'))"
        return f"CASE WHEN {aware} THEN date({q}, '+3 hours') ELSE date({q}) END"
    if encoding == "yyyymmdd":
        return f"date(substr({q},1,4)||'-'||substr({q},5,2)||'-'||substr({q},7,2))"
    scale = {"s": 1, "ms": 1000, "us": 1000000, "ns": 1000000000}[encoding]
    return f"date({q} / {scale}.0, 'unixepoch', '+3 hours')"


@contextmanager
def readonly_connection(path):
    p = Path(path).resolve()
    if not p.is_file():
        raise FileNotFoundError(f"SQLite database not found: {p}")
    conn = sqlite3.connect(p.as_uri() + "?mode=ro", uri=True)
    try:
        conn.execute("PRAGMA query_only=ON")
        yield conn
    finally:
        conn.close()


def actual_keys(config):
    return ["report_month", "report_dt", "dt_in", *config.segments]


def with_report_date(frame):
    if "report_dt" not in frame and "date" in frame:
        return frame.rename(columns={"date": "report_dt"})
    return frame


def load_raw(config, before=None, month=None):
    """SQL excludes target facts from the prediction read, not just from fitting."""
    if (before is None) == (month is None):
        raise ValueError("Specify either before or month")
    cols = [*actual_keys(config), "w1", "w2", "cession", "write_off", "target", "acc_count", "avg_dpd", "avg_pvr"]
    if config.raw_available_column:
        cols.append(config.raw_available_column)
    table = identifier(config.raw_table)
    expr = sql_date("report_dt", config.raw_date_encoding)
    with readonly_connection(config.db_path) as conn:
        present = {r[1] for r in conn.execute(f"PRAGMA table_info({table})")}
        missing = set(cols + ["is_forecast"]) - present
        if missing:
            raise ValueError(f"{config.raw_table}: missing columns {sorted(missing)}")
        invalid = conn.execute(
            f"SELECT COUNT(*) FROM {table} WHERE is_forecast=0 AND {expr} IS NULL"
        ).fetchone()[0]
        if invalid:
            raise ValueError(f"{invalid} invalid report_dt values; check raw_date_encoding")
        if before is not None:
            where, params = f"{expr} < ?", [month_start(before).strftime("%Y-%m-%d")]
        else:
            start = month_start(month)
            where = f"{expr} >= ? AND {expr} < ?"
            params = [start.strftime("%Y-%m-%d"), (start + pd.offsets.MonthBegin()).strftime("%Y-%m-%d")]
        frame = pd.read_sql_query(
            f"SELECT {', '.join(identifier(c) for c in cols)} FROM {table} "
            f"WHERE is_forecast=0 AND {where}", conn, params=params)
    return prepare_actual(frame, config, before=before, month=month)


def prepare_actual(frame, config, *, before=None, month=None):
    """Validate raw RR rows from either SQLite or the caller's DataFrame."""
    if (before is None) == (month is None):
        raise ValueError("Specify either before or month")
    frame = with_report_date(frame)
    dims = list(config.segments)
    precomputed = "fact" in frame
    amounts = ["fact"] if precomputed else ["w1", "w2", "cession", "write_off"]
    portfolio = ["acc_count", "avg_dpd", "avg_pvr"]
    if precomputed:
        portfolio.append("w1")
    columns = [*actual_keys(config), *amounts]
    if before is not None:
        columns.append("target")
    elif "target" in frame:
        columns.append("target")
    columns += portfolio if before is not None and not precomputed else [c for c in portfolio if c in frame]
    if config.raw_available_column:
        columns.append(config.raw_available_column)
    missing = set(columns) - set(frame)
    if missing:
        raise ValueError(f"Actual frame: missing columns {sorted(missing)}")
    if frame.empty:
        raise ValueError(f"No actual rows found for {'history before ' + str(before) if before else month}")
    if "is_forecast" in frame and not pd.to_numeric(frame.is_forecast, errors="raise").eq(0).all():
        raise ValueError("Actual frames must contain only rows with is_forecast=0")
    frame = frame[columns].copy()
    for col in ["report_dt", "report_month", "dt_in"]:
        frame[col] = parse_dates(frame[col], config.raw_date_encoding).dt.normalize()
        if frame[col].isna().any():
            raise ValueError(f"Missing date in actual frame: {col}")
    if before is not None and frame.report_dt.ge(month_start(before)).any():
        raise ValueError(f"df_history must contain only dates before {month_start(before).date()}")
    if month is not None and not frame.report_dt.dt.to_period("M").eq(pd.Period(month, "M")).all():
        raise ValueError(f"actual_df must contain only dates in {month}")
    frame = normalize_segments(frame, dims)
    if (frame.report_dt.dt.to_period("M") != frame.report_month.dt.to_period("M")).any():
        raise ValueError("report_month and report_dt disagree")
    if not frame.report_month.dt.day.eq(1).all():
        raise ValueError("report_month must be the first day of the month")
    if frame.dt_in.isna().any() or frame.dt_in.gt(frame.report_dt).any():
        raise ValueError("Invalid or future dt_in in an actual row")
    numeric(frame, [c for c in ["fact", "w1", "w2", "cession", "write_off", "target", "acc_count"] if c in frame])
    for col in ["avg_dpd", "avg_pvr"]:
        if col not in frame:
            continue
        frame[col] = pd.to_numeric(frame[col], errors="raise")
        if np.isinf(frame[col]).any():
            raise ValueError(f"Infinite values in {col}")
    if not precomputed:
        frame["fact"] = frame.w1 - frame.w2 - frame.cession - frame.write_off
    # Preserve source values: a snapshot may be filtered repeatedly at different origins.
    # Differencing must occur AFTER selecting the available vintage, never before it.
    frame["_source_fact"] = frame.fact
    if "target" in frame:
        frame["_source_target"] = frame.target
    frame["date"] = frame.report_dt
    frame["month"] = frame.report_dt.dt.strftime("%Y-%m")
    if config.raw_available_column:
        frame["available_at"] = parse_dates(frame[config.raw_available_column])
        if frame.available_at.isna().any():
            raise ValueError("Missing raw availability timestamp")
    key = ["report_month", "date", "dt_in", *dims]
    version_key = key + (["available_at"] if config.raw_available_column else [])
    if frame.duplicated(version_key).any():
        raise ValueError("Duplicate raw grain (including dt_in); resolve duplicates before aggregation")
    return frame


def cohort_daily(frame, value, mode, config):
    """Convert stable cohort snapshots without bridging gaps or erasing corrections."""
    if mode == "daily":
        return frame[value].copy()
    result = pd.Series(index=frame.index, dtype=float)
    month_ends = frame.groupby("report_month").date.max()
    keys = ["report_month", "dt_in", *config.segments]
    for _, group in frame.groupby(keys, sort=False, dropna=False):
        group = group.sort_values("date")
        start = max(group.report_month.iloc[0], group.dt_in.iloc[0])
        before_entry = group.date < start
        if group.loc[before_entry, value].ne(0).any():
            raise ValueError(f"Nonzero cumulative {value} before cohort entry")
        active = group.loc[~before_entry]
        end = month_ends.loc[group.report_month.iloc[0]]
        expected = pd.date_range(start, end, freq="D")
        if not pd.DatetimeIndex(active.date).equals(expected):
            raise ValueError(f"Cumulative {value}: missing cohort snapshots from {start.date()} to {end.date()}; "
                             "supply a complete stable cohort series or independently reconciled daily amounts")
        result.loc[group.index[before_entry]] = 0.0
        if not active.empty:
            change = active[value].diff()
            change.iloc[0] = active[value].iloc[0]
            result.loc[active.index] = change
    return result


def raw_asof(raw, config, cutoff):
    f = raw[raw.date <= cutoff].copy()
    if config.raw_available_column:
        f = f[f.available_at <= cutoff]
        key = ["report_month", "date", "dt_in", *config.segments]
        f = f.sort_values("available_at").drop_duplicates(key, keep="last")
    f = f.reset_index(drop=True)
    if "_source_fact" not in f:
        f["_source_fact"] = f.fact
    f["fact"] = cohort_daily(f, "_source_fact", config.fact_mode, config)
    if "target" in f:
        if "_source_target" not in f:
            f["_source_target"] = f.target
        f["target"] = cohort_daily(f, "_source_target", config.plan_mode, config)
    return f


def daily_actual(raw, config):
    raw = raw_asof(raw, config, pd.Timestamp.max - pd.Timedelta(days=1))
    aggregates = {"fact": ("fact", "sum")}
    if "target" in raw:
        aggregates["raw_plan"] = ("target", "sum")
    aggregates["raw_rows"] = ("fact", "size")
    return raw.groupby(["date", *config.segments], as_index=False, dropna=False).agg(**aggregates)


def load_plan(config, month):
    """Read target only: target-month fact/debt columns are never selected."""
    table_name = config.plan_table or config.raw_table
    table = identifier(table_name)
    source = [*actual_keys(config), config.plan_value_column]
    if config.plan_available_column:
        source.append(config.plan_available_column)
    start = month_start(month)
    end = start + pd.offsets.MonthBegin()
    date_expr = sql_date("report_dt", config.raw_date_encoding)
    with readonly_connection(config.db_path) as conn:
        present = {r[1] for r in conn.execute(f"PRAGMA table_info({table})")}
        missing = set([*source, "is_forecast"]) - present
        if missing:
            raise ValueError(f"Plan table {table_name}: missing columns {sorted(missing)}")
        frame = pd.read_sql_query(
            f"SELECT {', '.join(identifier(c) for c in source)} FROM {table} "
            f"WHERE is_forecast=? AND {date_expr} >= ? AND {date_expr} < ?",
            conn, params=[config.plan_is_forecast, start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")])
    return aggregate_plan(prepare_plan(frame, config), config, month)


def prepare_plan(frame, config):
    frame = with_report_date(frame)
    columns = [*actual_keys(config), config.plan_value_column]
    rename = {config.plan_value_column: "plan"}
    if config.plan_available_column:
        columns.append(config.plan_available_column)
        rename[config.plan_available_column] = "plan_available_at"
    missing = set(columns) - set(frame)
    if missing:
        raise ValueError(f"Plan frame: missing columns {sorted(missing)}")
    if "is_forecast" in frame:
        kinds = pd.to_numeric(frame.is_forecast, errors="raise")
        if not kinds.isin([0, 1]).all():
            raise ValueError("Plan is_forecast values must be 0 or 1")
        frame = frame.loc[kinds.eq(config.plan_is_forecast)].copy()
    frame = normalize_segments(frame[columns].rename(columns=rename), config.segments)
    for col in ["report_month", "report_dt", "dt_in"]:
        frame[col] = parse_dates(frame[col], config.raw_date_encoding).dt.normalize()
        if frame[col].isna().any():
            raise ValueError(f"Missing date in plan: {col}")
    if not frame.report_month.eq(frame.report_dt.dt.to_period("M").dt.start_time).all():
        raise ValueError("Plan report_month and report_dt disagree")
    if "plan_available_at" in frame:
        frame["plan_available_at"] = parse_dates(frame.plan_available_at)
        if frame.plan_available_at.isna().any():
            raise ValueError("Missing plan availability timestamp")
    return frame


def split_dated_frame(df, config):
    """Adapt date/fact/plan rows to separate history, plan and evaluation inputs."""
    missing = {"date", "fact", "plan"} - set(df)
    if missing:
        raise ValueError(f"run(df) needs columns date, fact, plan. Missing: {sorted(missing)}")
    frame = df.copy()
    frame["report_dt"] = parse_dates(frame.date, config.raw_date_encoding).dt.normalize()
    if frame.report_dt.isna().any():
        raise ValueError("Missing date in run(df) input")
    frame["report_month"] = frame.report_dt.dt.to_period("M").dt.start_time
    if "dt_in" not in frame:
        frame["dt_in"] = frame.report_month
    for col in config.segments:
        if col not in frame:
            frame[col] = "0" if col == "inflow" else "ВСЕ"
    frame["target"] = frame.plan
    start = month_start(config.forecast_month)
    end = start + pd.offsets.MonthBegin()
    history = frame.loc[frame.report_dt.lt(start) & frame.fact.notna()]
    plan_cols = [*actual_keys(config), "target"]
    if config.plan_available_column:
        plan_cols.append(config.plan_available_column)
    plan = frame.loc[frame.report_dt.lt(end) & frame.target.notna(), plan_cols]
    actual = frame.loc[frame.report_dt.ge(start) & frame.report_dt.lt(end) & frame.fact.notna()]
    return history, plan, actual


def aggregate_plan(rows, config, month):
    dims = list(config.segments)
    frame = rows.loc[rows.report_month.eq(month_start(month))].copy()
    if frame.empty:
        raise ValueError(f"No daily plan for {month}")
    frame["date"] = frame.report_dt
    key = ["date", "dt_in", *dims]
    if "plan_available_at" in frame:
        frame = frame[frame.plan_available_at <= origin(month)]
        if frame.empty:
            raise ValueError(f"No plan version available before {month}")
        if frame.duplicated([*key, "plan_available_at"]).any():
            raise ValueError("Duplicate plan cohort and availability timestamp")
        frame = frame.sort_values("plan_available_at").drop_duplicates(key, keep="last")
    elif frame.duplicated(key).any():
        raise ValueError("Duplicate daily plan cohort; do not sum repeated snapshots")
    numeric(frame, ["plan"])
    frame = frame.reset_index(drop=True)
    frame["plan"] = cohort_daily(frame, "plan", config.plan_mode, config)
    if frame.plan.lt(0).any():
        raise ValueError("Daily plans must be nonnegative; resolve corrections explicitly")
    # dt_in is retained until here: each cohort contributes once to its segment/day.
    result = frame.groupby(["date", *dims], as_index=False).plan.sum()
    first_entry = frame.groupby(dims, as_index=False).dt_in.min()
    expected_days = days(month)
    # Before the first planned inflow cohort exists its plan is structurally zero.
    # Other absent days remain errors, never guessed from payment observations.
    added = []
    for key_values, part in result.groupby(dims, sort=False):
        key_values = key_values if isinstance(key_values, tuple) else (key_values,)
        leaf = dict(zip(dims, key_values))
        missing_days = expected_days.difference(part.date)
        if len(missing_days) and leaf.get("inflow") == "1":
            entry = first_entry
            for col, value in leaf.items():
                entry = entry[entry[col].eq(value)]
            first_day = entry.dt_in.iloc[0]
            structural = missing_days[missing_days < first_day]
            added.extend({"date": d, **leaf, "plan": 0.0} for d in structural)
    if added:
        result = pd.concat([result, pd.DataFrame(added)], ignore_index=True)
    counts = result.groupby(dims).date.nunique()
    if not counts.eq(len(expected_days)).all():
        raise ValueError(f"{month}: every segment needs a plan for every day; missing plan rows are not assumed zero")
    if result.plan.sum() <= 0:
        raise ValueError(f"{month}: the entire monthly plan is zero; efficiency cannot be assessed")
    return result.sort_values([*dims, "date"]).reset_index(drop=True)


def read_input(path, required, config):
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"Input not found: {p}. See input/*_template.csv and README.md")
    f = pd.read_parquet(p) if p.suffix.lower() == ".parquet" else pd.read_csv(p, dtype={c: str for c in config.segments})
    missing = set(required) - set(f)
    if missing:
        raise ValueError(f"{p.name}: missing columns {sorted(missing)}")
    if f.empty:
        raise ValueError(f"{p.name} is empty")
    f = normalize_segments(f, config.segments)
    for c in ["date", "known_at", "snapshot_date"]:
        if c in f:
            f[c] = parse_dates(f[c])
            if f[c].isna().any():
                raise ValueError(f"{p.name}: missing {c}")
            if c != "known_at":
                f[c] = f[c].dt.normalize()
    if "month" in f:
        f["month"] = pd.to_datetime(f["month"], format="mixed", errors="raise").dt.strftime("%Y-%m")
    return f


class Inputs:
    def __init__(self, config, *, plan_df=None):
        self.config = config
        dims = list(config.segments)
        self._plan_cache = {}
        self._plan_rows = None if plan_df is None else prepare_plan(plan_df, config)
        self.portfolio = self._optional(config.portfolio_path,
            ["month", "snapshot_date", "known_at", "debt", "acc_count", *dims])
        self.inflow = self._optional(config.inflow_path,
            ["month", "date", "known_at", "incoming_debt", *dims])
        self.operations = self._optional(config.operations_path, ["date", "known_at", *dims])
        self.reference = self._optional(config.reference_path,
            ["month", "date", "known_at", "forecast_daily", *dims])
        for frame, columns in [(self.portfolio, ["debt", "acc_count"]),
                               (self.inflow, ["incoming_debt"]), (self.reference, ["forecast_daily"])]:
            if frame is not None:
                numeric(frame, columns)
                if "date" in frame and (frame.date.dt.strftime("%Y-%m") != frame.month).any():
                    raise ValueError("Input month and date disagree")
        if self.operations is not None:
            opcols = [c for c in self.operations if c.startswith("op_")]
            if not opcols:
                raise ValueError("Operations input needs numeric op_* columns")
            numeric(self.operations, opcols)
        self.calendar = None
        if config.calendar_path:
            self.calendar = pd.read_csv(config.calendar_path)
            if not {"date", "is_workday"} <= set(self.calendar):
                raise ValueError("Calendar needs date,is_workday columns")
            self.calendar["date"] = parse_dates(self.calendar.date).dt.normalize()
            if self.calendar.date.duplicated().any() or not self.calendar.is_workday.isin([0, 1]).all():
                raise ValueError("Calendar dates must be unique, is_workday must be 0/1")
            if "is_holiday" in self.calendar and not self.calendar.is_holiday.isin([0, 1]).all():
                raise ValueError("Calendar is_holiday must be 0/1")
        elif config.require_calendar_overrides:
            raise ValueError("Provide calendar_path with the approved production calendar, including transfers")

    def _optional(self, path, required):
        return read_input(path, required, self.config) if path else None

    def vintage(self, frame, month, value_columns, extra_keys=()):
        if frame is None:
            return None
        f = frame[(frame.month == str(month)) & (frame.known_at <= origin(month))].copy()
        key = [*self.config.segments, *extra_keys]
        if f.duplicated([*key, "known_at"]).any():
            raise ValueError("Duplicate input grain and known_at; aggregate cohorts before supplying input")
        f = f.sort_values("known_at").drop_duplicates(key, keep="last")
        return f[[*key, *value_columns]]

    def plan(self, month):
        month = str(month)
        if month not in self._plan_cache:
            if self._plan_rows is None:
                self._plan_cache[month] = load_plan(self.config, month)
            else:
                self._plan_cache[month] = aggregate_plan(self._plan_rows, self.config, month)
        return self._plan_cache[month].copy()


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


## features

In [ ]:
import numpy as np
import pandas as pd


class FeatureBuilder:
    def __init__(self, config, inputs):
        self.config, self.inputs = config, inputs
        self._calendar_cache = {}
        self._feature_cache = {}

    def calendar(self, month):
        month = str(month)
        if month in self._calendar_cache:
            return self._calendar_cache[month].copy()
        dates = days(month)
        start, end = dates[0], dates[-1]
        extended = pd.date_range(start - pd.Timedelta(days=40), end + pd.Timedelta(days=40))
        try:
            import holidays
            holiday_set = holidays.country_holidays(self.config.country, years=sorted(set(extended.year)))
            holiday = pd.Series([int(d.date() in holiday_set) for d in extended], index=extended)
        except ImportError:
            raise ImportError("Install holidays from requirements.txt to construct calendar features")
        workday = pd.Series([int(holiday_set.is_working_day(d.date())) for d in extended], index=extended)
        overrides = self.inputs.calendar
        if overrides is not None:
            indexed = overrides.set_index("date")
            if self.config.require_calendar_overrides and not dates.isin(indexed.index).all():
                raise ValueError(f"Production calendar is incomplete for {month}")
            common = extended.intersection(indexed.index)
            workday.loc[common] = indexed.loc[common, "is_workday"].astype(int)
            if "is_holiday" in indexed:
                holiday.loc[common] = indexed.loc[common, "is_holiday"].astype(int)
        f = pd.DataFrame({"date": dates})
        f["day"] = dates.day
        f["weekday"] = dates.day_of_week
        f["month_number"] = start.month
        f["year_index"] = start.year - 2024
        f["days_in_month"] = len(dates)
        f["days_left"] = len(dates) - f.day
        f["week_of_month"] = (f.day - 1) // 7
        f["month_progress"] = f.day / len(dates)
        f["is_weekend"] = (f.weekday >= 5).astype(int)
        f["is_workday"] = workday.loc[dates].to_numpy()
        f["is_holiday"] = holiday.loc[dates].to_numpy()
        f["pre_holiday"] = holiday.loc[dates + pd.Timedelta(days=1)].to_numpy()
        f["post_holiday"] = holiday.loc[dates - pd.Timedelta(days=1)].to_numpy()
        before, after = {}, {}
        streak = 0
        for date, is_working in workday.items():
            before[date] = streak
            streak = 0 if is_working else streak + 1
        streak = 0
        for date, is_working in workday.iloc[::-1].items():
            after[date] = streak
            streak = 0 if is_working else streak + 1
        f["nonwork_before"] = [before[d] for d in dates]
        f["nonwork_after"] = [after[d] for d in dates]
        f["workdays_left"] = f.is_workday.iloc[::-1].cumsum().iloc[::-1].to_numpy()
        f["workday_number"] = f.is_workday.cumsum()
        f["first_workday"] = (f.is_workday.eq(1) & f.workday_number.eq(1)).astype(int)
        f["last_workday"] = (f.is_workday.eq(1) & f.workdays_left.eq(1)).astype(int)
        f["month_start_flag"] = (f.day <= 5).astype(int)
        f["month_end_flag"] = (f.days_left < 5).astype(int)
        f["year_start_flag"] = int(start.month == 1)
        f["year_end_flag"] = int(start.month == 12)
        f["annual_sin"] = np.sin(2 * np.pi * dates.day_of_year / 365.25)
        f["annual_cos"] = np.cos(2 * np.pi * dates.day_of_year / 365.25)
        self._calendar_cache[month] = f
        return f

    def portfolio(self, history, month):
        dims = list(self.config.segments)
        external = self.inputs.portfolio
        if external is not None:
            f = external[(external.month == str(month)) & (external.known_at <= origin(month))
                         & (external.snapshot_date <= origin(month))].copy()
            if f.empty:
                if history.empty:
                    return pd.DataFrame(columns=[*dims, "debt", "acc_count", "avg_dpd", "avg_pvr", "snapshot_age_days"])
                raise ValueError(f"No portfolio snapshot available before {month}")
            if f.duplicated([*dims, "known_at"]).any():
                raise ValueError("Duplicate portfolio vintage")
            f = f.sort_values(["snapshot_date", "known_at"]).drop_duplicates(dims, keep="last")
            for c in ["avg_dpd", "avg_pvr"]:
                if c not in f:
                    f[c] = np.nan
                f[c] = pd.to_numeric(f[c], errors="raise")
            f["snapshot_age_days"] = (month_start(month) - f.snapshot_date).dt.days
            return f[[*dims, "debt", "acc_count", "avg_dpd", "avg_pvr", "snapshot_age_days"]]
        h = raw_asof(history, self.config, origin(month))
        if h.empty or not {"w1", "acc_count", "avg_dpd", "avg_pvr"} <= set(h):
            # The first historical training month may have no earlier snapshot.
            # Mark missing initial state; never substitute that month's observed portfolio.
            return pd.DataFrame(columns=[*dims, "debt", "acc_count", "avg_dpd", "avg_pvr", "snapshot_age_days"])
        latest = h.date.max()
        h = h[h.date == latest].copy()
        h["dpd_weighted"] = h.avg_dpd * h.w1
        h["pvr_weighted"] = h.avg_pvr * h.w1
        h["dpd_debt"] = h.w1.where(h.avg_dpd.notna(), 0)
        h["pvr_debt"] = h.w1.where(h.avg_pvr.notna(), 0)
        f = h.groupby(dims, as_index=False).agg(
            debt=("w1", "sum"), acc_count=("acc_count", "sum"),
            dpd_weighted=("dpd_weighted", "sum"), pvr_weighted=("pvr_weighted", "sum"),
            dpd_debt=("dpd_debt", "sum"), pvr_debt=("pvr_debt", "sum"))
        f["avg_dpd"] = f.dpd_weighted / f.dpd_debt.replace(0, np.nan)
        f["avg_pvr"] = f.pvr_weighted / f.pvr_debt.replace(0, np.nan)
        f["snapshot_age_days"] = (month_start(month) - latest).days
        return f[[*dims, "debt", "acc_count", "avg_dpd", "avg_pvr", "snapshot_age_days"]]

    def build(self, history, month):
        """All observed features are frozen before month, including in training."""
        cache_key = (id(history), str(month))
        if cache_key in self._feature_cache:
            return self._feature_cache[cache_key].copy()
        dims = list(self.config.segments)
        h = raw_asof(history, self.config, origin(month))
        f = self.inputs.plan(month)
        f = f.merge(self.calendar(month), on="date", validate="many_to_one")
        f = f.sort_values([*dims, "date"]).reset_index(drop=True)
        f["plan_cumulative"] = f.groupby(dims).plan.cumsum()
        f["plan_month"] = f.groupby(dims).plan.transform("sum")
        f["plan_daily_share"] = f.plan / f.plan_month.replace(0, np.nan)
        f["plan_cumulative_share"] = f.plan_cumulative / f.plan_month.replace(0, np.nan)
        portfolio = self.portfolio(h, month)
        absent = portfolio[dims].merge(f[dims].drop_duplicates(), how="left", indicator=True, on=dims)
        if absent._merge.eq("left_only").any():
            raise ValueError(f"{month}: portfolio segments missing from plan; supply explicit zero-plan rows")
        f = f.merge(portfolio, how="left", on=dims, validate="many_to_one")
        f["new_segment"] = f.debt.isna().astype(int)
        f["debt"] = f.debt.fillna(0)
        f["acc_count"] = f.acc_count.fillna(0)
        f["average_debt"] = f.debt / f.acc_count.replace(0, np.nan)
        f["portfolio_total_debt"] = portfolio.debt.sum()
        f["portfolio_debt_share"] = f.debt / max(float(portfolio.debt.sum()), 1.0)
        for n in (1, 3, 6):
            cutoff = month_start(month) - pd.DateOffset(months=n)
            part = h[h.date >= cutoff]
            stats = part.groupby(dims, as_index=False).agg(
                past_fact=("fact", "sum"), past_plan=("target", "sum"))
            stats[f"history_eff_{n}m"] = stats.past_fact / stats.past_plan.replace(0, np.nan)
            stats = stats.rename(columns={"past_fact": f"history_fact_{n}m", "past_plan": f"history_plan_{n}m"})
            f = f.merge(stats, on=dims, how="left", validate="many_to_one")
            denominator = part.target.sum()
            f[f"total_history_eff_{n}m"] = part.fact.sum() / denominator if denominator else np.nan
        for n in (7, 14, 28):
            part = h[h.date >= month_start(month) - pd.Timedelta(days=n)]
            stats = part.groupby(dims, as_index=False).agg(recent_fact=("fact", "sum"),
                                                         observed_days=("date", "nunique"))
            stats["recent_fact"] = stats.recent_fact.where(stats.observed_days.eq(n))
            stats = stats.rename(columns={"recent_fact": f"recent_fact_{n}d", "observed_days": f"recent_days_{n}d"})
            f = f.merge(stats, on=dims, how="left", validate="many_to_one")
        inflow = self.inputs.vintage(self.inputs.inflow, month, ["incoming_debt"], ["date"])
        f["incoming_debt"] = 0.0
        if inflow is not None:
            unknown = inflow[dims].drop_duplicates().merge(f[dims].drop_duplicates(), on=dims, how="left", indicator=True)
            if unknown._merge.eq("left_only").any():
                raise ValueError("Planned inflow has segments missing from plan")
            f = f.drop(columns="incoming_debt").merge(inflow, on=[*dims, "date"], how="left", validate="one_to_one")
            # Inflow input is a sparse schedule; absent entries mean no scheduled inflow.
            f["incoming_debt"] = f.incoming_debt.fillna(0)
        f["incoming_debt_cumulative"] = f.groupby(dims).incoming_debt.cumsum()
        if self.inputs.operations is not None:
            ops = self.inputs.operations
            ops = ops[(ops.date < month_start(month)) & (ops.known_at <= origin(month))]
            key = [*dims, "date"]
            if ops.duplicated([*key, "known_at"]).any():
                raise ValueError("Duplicate operations vintage")
            ops = ops.sort_values("known_at").drop_duplicates(key, keep="last")
            for n in (7, 30):
                part = ops[ops.date >= month_start(month) - pd.Timedelta(days=n)]
                cols = [c for c in self.inputs.operations if c.startswith("op_")]
                summed = part.groupby(dims, as_index=False)[cols].sum()
                summed = summed.rename(columns={c: f"{c}_{n}d" for c in cols})
                f = f.merge(summed, on=dims, how="left", validate="many_to_one")
        for col in f:
            if col not in [*dims, "date"]:
                f[col] = pd.to_numeric(f[col], errors="raise").replace([np.inf, -np.inf], np.nan).fillna(-1.0)
        self._feature_cache[cache_key] = f
        return f.copy()

    def total(self, features):
        dims = list(self.config.segments)
        calendar = self.calendar(features.date.iloc[0].strftime("%Y-%m"))
        sums = ["plan", "plan_cumulative", "plan_month", "debt", "acc_count", "incoming_debt", "incoming_debt_cumulative"]
        t = features.groupby("date", as_index=False)[sums].sum().merge(calendar, on="date", validate="one_to_one")
        t["plan_daily_share"] = t.plan / t.plan_month.replace(0, np.nan)
        t["plan_cumulative_share"] = t.plan_cumulative / t.plan_month.replace(0, np.nan)
        for col in features:
            if col.startswith("total_history_"):
                t[col] = features.groupby("date")[col].first().reindex(t.date).to_numpy()
        for col in ["avg_dpd", "avg_pvr"]:
            valid = features[col].ge(0)
            num = (features[col] * features.debt.where(valid, 0)).groupby(features.date).sum()
            den = features.debt.where(valid, 0).groupby(features.date).sum()
            t[col] = (num / den.replace(0, np.nan)).reindex(t.date).to_numpy()
        for col in features:
            if col.startswith("op_"):
                t[col] = features.groupby("date")[col].sum().reindex(t.date).to_numpy()
        return t.replace([np.inf, -np.inf], np.nan).fillna(-1.0)


## models

In [ ]:
"""Candidate models fitted without observations from the forecast month."""
import warnings
import numpy as np
import pandas as pd


def label_features(features, actual, config):
    dims = list(config.segments)
    f = features.merge(actual[["date", *dims, "fact"]], on=["date", *dims], how="outer", validate="one_to_one", indicator=True)
    if f.fact.isna().any() or f._merge.ne("both").any():
        return None
    f = f.drop(columns="_merge").sort_values([*dims, "date"]).reset_index(drop=True)
    f["fact_cumulative"] = f.groupby(dims).fact.cumsum()
    return f


def training_data(history, month, config, builder):
    available = raw_asof(history, config, origin(month))
    actual = daily_actual(available, config)
    actual["month"] = actual.date.dt.strftime("%Y-%m")
    frames, totals, skipped = [], [], []
    for previous in sorted(available.month.unique()):
        if previous >= str(month):
            continue
        try:
            x = builder.build(history, previous)
            a = actual[actual.month == previous]
            f = label_features(x, a, config)
            if f is None:
                raise ValueError("missing segment-day facts; zeros were not imputed")
            total = builder.total(x)
            actual_total = a.groupby("date").fact.sum().reindex(total.date)
            if actual_total.isna().any():
                raise ValueError("missing daily total")
            total["fact"] = actual_total.to_numpy()
            total["fact_cumulative"] = total.fact.cumsum()
            total["efficiency"] = total.fact_cumulative / total.plan_cumulative.replace(0, np.nan)
            f["training_month"] = previous
            total["training_month"] = previous
            frames.append(f)
            totals.append(total)
        except ValueError as exc:
            skipped.append({"month": previous, "reason": str(exc)})
    if not frames:
        raise ValueError(f"No complete training months before {month}; inspect plan archives and fact coverage")
    return pd.concat(frames, ignore_index=True), pd.concat(totals, ignore_index=True), skipped


def profile_forecast(history, features, config, calendar=False, builder=None):
    dims = list(config.segments)
    h = raw_asof(history, config, origin(features.date.iloc[0].strftime("%Y-%m")))
    h = h[h.date >= features.date.min() - pd.DateOffset(months=config.reference_months)].copy()
    if h.empty:
        raise ValueError("Reference profile has no historical data")
    if not calendar:
        return normalized_reference(h, features, config)
    h["weekday"] = h.date.dt.day_of_week
    h["day"] = h.date.dt.day
    h["month_start_flag"] = h.day.le(5).astype(int)
    # Holiday information is supplied from the same production calendar used by ML.
    if calendar:
        cal = pd.concat([builder.calendar(m) for m in sorted(h.month.unique())])
        h = h.merge(cal[["date", "is_workday", "is_holiday"]], on="date", validate="many_to_one")
        keys = [*dims, "weekday", "month_start_flag", "is_workday", "is_holiday"]
    else:
        keys = [*dims, "day"]
    grouped = h.groupby(keys, as_index=False).agg(num=("fact", "sum"), den=("target", "sum"))
    grouped["ratio"] = grouped.num / grouped.den.replace(0, np.nan)
    segment = h.groupby(dims, as_index=False).agg(num=("fact", "sum"), den=("target", "sum"))
    segment["fallback_ratio"] = segment.num / segment.den.replace(0, np.nan)
    global_ratio = h.fact.sum() / h.target.sum() if h.target.sum() else 1.0
    f = features.merge(grouped[[*keys, "ratio"]], on=keys, how="left", validate="many_to_one")
    f = f.merge(segment[[*dims, "fallback_ratio"]], on=dims, how="left", validate="many_to_one")
    # Extrapolate a segment's recent execution level using a known future daily plan.
    ratio = f.ratio.fillna(f.fallback_ratio).fillna(global_ratio)
    predicted_daily = f.plan * ratio
    return predicted_daily.groupby([f[c] for c in dims]).cumsum().to_numpy()


def normalized_reference(history, features, config):
    """Interpolate cumulative execution on month progress, preserving the last value."""
    dims = list(config.segments)
    h = daily_actual(history, config)
    h["month"] = h.date.dt.strftime("%Y-%m")
    def curves(part, n):
        result = []
        for month, group in part.groupby("month", sort=True):
            group = group.groupby("date", as_index=False)[["fact", "raw_plan"]].sum().sort_values("date")
            if not pd.DatetimeIndex(group.date).equals(days(month)) or group.raw_plan.sum() <= 0:
                continue
            cumulative = np.r_[0.0, group.fact.cumsum().to_numpy() / group.raw_plan.sum()]
            result.append(np.interp(np.arange(1, n + 1) / n, np.linspace(0, 1, len(cumulative)), cumulative))
        return np.mean(result, axis=0) if result else None
    n = features.date.nunique()
    fallback = curves(h, n)
    if fallback is None:
        raise ValueError("No complete reference month; missing facts were not replaced with zeros")
    prediction = pd.Series(index=features.index, dtype=float)
    for key, group in features.groupby(dims, sort=False):
        key = key if isinstance(key, tuple) else (key,)
        part = h
        for col, value in zip(dims, key):
            part = part[part[col].eq(value)]
        shape = curves(part, n)
        group = group.sort_values("date")
        prediction.loc[group.index] = (fallback if shape is None else shape) * group.plan_month.iloc[0]
    return prediction.to_numpy()


def calendar_design(frame):
    """A small regularized calendar regression, also valid for signed adjustments."""
    columns = [np.ones(len(frame))]
    for field, values in [("day", range(1, 32)), ("weekday", range(7)), ("month_number", range(1, 13))]:
        columns.extend(frame[field].eq(v).to_numpy(dtype=float) for v in values)
    for field in ["is_workday", "is_holiday", "month_start_flag", "month_end_flag",
                  "pre_holiday", "post_holiday", "plan_daily_share", "total_history_eff_3m"]:
        columns.append(frame[field].to_numpy(dtype=float))
    return np.column_stack(columns)


def calendar_regression(train, future):
    matrix, target = calendar_design(train), train.fact.to_numpy() / train.plan_month.clip(lower=1).to_numpy()
    ridge = np.eye(matrix.shape[1]) * 10.0
    ridge[0, 0] = 1e-6
    coefficients = np.linalg.solve(matrix.T @ matrix + ridge, matrix.T @ target)
    daily = calendar_design(future) @ coefficients * future.plan_month.to_numpy()
    return np.cumsum(daily)


def _matrix(frame, columns):
    return frame[columns].copy()


def candidates(history, month, config, builder):
    x = builder.build(history, month)
    total_x = builder.total(x)
    leaves, totals, status = {}, {}, []
    leaves["reference_profile"] = profile_forecast(history, x, config)
    leaves["calendar_profile"] = profile_forecast(history, x, config, calendar=True, builder=builder)
    if builder.inputs.reference is not None:
        ext = builder.inputs.vintage(builder.inputs.reference, month, ["forecast_daily"], ["date"])
        joined = x[["date", *config.segments]].merge(ext, on=["date", *config.segments], how="left", validate="one_to_one")
        if joined.forecast_daily.isna().any():
            status.append({"model": "external_reference", "status": "unavailable", "reason": "incomplete pre-origin reference"})
        else:
            leaves["external_reference"] = joined.groupby(list(config.segments)).forecast_daily.cumsum().to_numpy()
    train, train_total, skipped = training_data(history, month, config, builder)
    status.extend({"model": "training_month", "status": "excluded", **item} for item in skipped)
    if train.training_month.nunique() < config.min_train_months:
        raise ValueError(f"Only {train.training_month.nunique()} complete training months before {month}; need {config.min_train_months}")
    totals["total_calendar_regression"] = calendar_regression(train_total, total_x)
    recent = train_total[train_total.training_month.isin(sorted(train_total.training_month.unique())[-config.reference_months:])]
    by_day = recent.groupby("day").efficiency.median()
    fallback = recent.efficiency.median()
    if not np.isfinite(fallback):
        fallback = 1.0
    totals["total_reference"] = total_x.day.map(by_day).fillna(fallback).to_numpy() * total_x.plan_cumulative.to_numpy()
    if config.enable_catboost:
        try:
            from catboost import CatBoostRegressor
        except ImportError:
            status.append({"model": "catboost", "status": "unavailable", "reason": "dependency not installed"})
        else:
            cols = [c for c in x if c != "date"]
            total_cols = [c for c in total_x if c != "date"]
            for depth in config.catboost_depths:
                params = dict(iterations=config.catboost_iterations, depth=depth, learning_rate=0.05,
                              l2_leaf_reg=10, loss_function="RMSE", random_seed=config.seed,
                              thread_count=1, verbose=False, allow_writing_files=False)
                for mode, target in [("daily", "fact"), ("cumulative", "fact_cumulative")]:
                    name = f"catboost_{mode}_d{depth}"
                    model = CatBoostRegressor(**params)
                    # Equalize monthly monetary scales without deriving any target-month features.
                    scale = train.plan_month.clip(lower=1.0)
                    model.fit(_matrix(train, cols), train[target] / scale, cat_features=list(config.segments))
                    pred = model.predict(_matrix(x, cols)) * x.plan_month.clip(lower=1.0).to_numpy()
                    if mode == "daily":
                        pred = pd.Series(pred).groupby([x[c] for c in config.segments]).cumsum().to_numpy()
                    leaves[name] = pred
                valid = train_total.efficiency.notna()
                model = CatBoostRegressor(**params)
                model.fit(_matrix(train_total.loc[valid], total_cols), train_total.loc[valid, "efficiency"])
                totals[f"total_catboost_d{depth}"] = model.predict(_matrix(total_x, total_cols)) * total_x.plan_cumulative.to_numpy()
    if config.enable_sarimax:
        try:
            totals["total_sarimax"] = sarimax_forecast(history, total_x, month, config, builder)
        except (ImportError, ValueError, np.linalg.LinAlgError) as exc:
            status.append({"model": "total_sarimax", "status": "unavailable", "reason": str(exc)})
    for name, pred in [*leaves.items(), *totals.items()]:
        if not np.isfinite(pred).all():
            raise ValueError(f"Nonfinite predictions: {name}")
        status.append({"model": name, "status": "available", "month": str(month)})
    return x, leaves, totals, status


def sarimax_forecast(history, total_x, month, config, builder):
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    h = raw_asof(history, config, origin(month))
    dates = pd.date_range(h.date.min(), month_start(month) - pd.Timedelta(days=1), freq="D")
    a = h.groupby("date")[["fact", "target"]].sum().reindex(dates)
    if a.isna().any().any():
        raise ValueError("SARIMAX needs complete daily history; missing days were not imputed")
    calendars = pd.concat([builder.calendar(m) for m in pd.period_range(dates.min(), dates.max(), freq="M")])
    c = calendars.set_index("date").reindex(dates)
    columns = ["month_start_flag", "month_end_flag", "is_workday", "is_holiday", "annual_sin", "annual_cos"]
    past = c[columns].astype(float).copy()
    future = total_x[columns].astype(float).copy()
    for wd in range(1, 7):
        past[f"weekday_{wd}"] = (dates.day_of_week == wd).astype(float)
        future[f"weekday_{wd}"] = total_x.weekday.eq(wd).astype(float)
    plan_scale = max(float(a.target.abs().mean()), 1.0)
    fact_scale = max(float(a.fact.abs().mean()), 1.0)
    past["plan"] = a.target.to_numpy() / plan_scale
    future["plan"] = total_x.plan.to_numpy() / plan_scale
    # Drop constant regressors using training data only (a SARIMAX intercept is included).
    usable = past.columns[past.nunique() > 1]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fitted = SARIMAX(a.fact.to_numpy() / fact_scale, exog=past[usable].to_numpy(),
                         order=(1, 0, 1), seasonal_order=(1, 0, 0, 7), trend="c",
                         enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=150)
    if not fitted.mle_retvals.get("converged", False):
        raise ValueError("SARIMAX optimizer did not converge")
    return np.cumsum(np.asarray(fitted.get_forecast(len(total_x), exog=future[usable].to_numpy()).predicted_mean) * fact_scale)


def specs(leaves, totals, weights):
    for leaf in sorted(leaves):
        yield {"leaf": leaf, "total": None, "weight": 0.0}
        for total in sorted(totals):
            for weight in weights:
                if weight > 0:
                    yield {"leaf": leaf, "total": total, "weight": float(weight)}


def spec_id(spec):
    return f"{spec['leaf']}|{spec['total'] or 'bottom_up'}|{spec['weight']:g}"


def reconcile(features, leaf_cumulative, total_cumulative, weight, config):
    dims = list(config.segments)
    f = features[["date", *dims, "plan", "plan_cumulative", "plan_month", "debt"]].copy()
    f["base_forecast_cumulative"] = np.asarray(leaf_cumulative, dtype=float)
    bottom = f.groupby("date").base_forecast_cumulative.sum()
    direct = bottom.to_numpy() if total_cumulative is None else np.asarray(total_cumulative)
    chosen = (1 - weight) * bottom.to_numpy() + weight * direct
    target = pd.Series(chosen, index=bottom.index)
    daily_den = f.groupby("date").plan_cumulative.transform("sum")
    monthly_den = f.groupby("date").plan_month.transform("sum")
    debt = f.debt.clip(lower=0)
    debt_den = debt.groupby(f.date).transform("sum")
    count = f.groupby("date").date.transform("size")
    alpha = np.where(daily_den > 0, f.plan_cumulative / daily_den.replace(0, np.nan),
                     np.where(monthly_den > 0, f.plan_month / monthly_den.replace(0, np.nan),
                              np.where(debt_den > 0, debt / debt_den.replace(0, np.nan), 1 / count)))
    delta = f.date.map(target - bottom).to_numpy()
    f["reconciliation_cumulative"] = alpha * delta
    f["forecast_cumulative"] = f.base_forecast_cumulative + f.reconciliation_cumulative
    f = f.sort_values([*dims, "date"]).reset_index(drop=True)
    for cumulative, daily in [("forecast_cumulative", "forecast_daily"),
                              ("reconciliation_cumulative", "reconciliation_daily")]:
        f[daily] = f.groupby(dims)[cumulative].diff().fillna(f[cumulative])
    summed = f.groupby("date").forecast_cumulative.sum().reindex(target.index)
    if not np.allclose(summed, target, rtol=1e-10, atol=1e-7):
        raise AssertionError("Reconciliation does not preserve the total")
    f["origin"] = origin(f.date.iloc[0].strftime("%Y-%m"))
    return f


## pipeline

In [ ]:
import hashlib
import importlib.metadata
import json
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd


def evaluate(forecast, actual, config):
    """Evaluation is the only API consuming target-month actuals."""
    keys = ["date", *config.segments]
    if forecast.empty or forecast.duplicated(keys).any():
        raise ValueError("Forecast must have a unique, nonempty segment-day grid")
    forecast = forecast.copy()
    numeric(forecast, ["plan", "forecast_daily"])
    if forecast.date.dt.to_period("M").nunique() != 1:
        raise ValueError("Evaluate one forecast month at a time")
    expected = days(forecast.date.iloc[0].strftime("%Y-%m"))
    for _, group in forecast.groupby(list(config.segments)):
        if not pd.DatetimeIndex(group.date.sort_values()).equals(expected):
            raise ValueError("Forecast grid must cover every date of the month for every segment")
    if not actual.date.isin(expected).all():
        raise ValueError("Actual dates fall outside the forecast month")
    if actual.duplicated(keys).any():
        raise ValueError("Actuals must be aggregated to the forecast grain")
    f = forecast.merge(actual[[*keys, "fact"]], on=keys, how="outer", indicator=True, validate="one_to_one")
    f["missing_actual"] = f._merge.eq("left_only") | f.fact.isna()
    f["unplanned_actual"] = f._merge.eq("right_only")
    # Extra actual segments must contribute to the total, never disappear in an inner join.
    for c in ["plan", "forecast_daily", "reconciliation_daily"]:
        f.loc[f.unplanned_actual, c] = 0.0
    f = f.sort_values([*config.segments, "date"]).reset_index(drop=True)
    f["forecast_cumulative"] = f.groupby(list(config.segments)).forecast_daily.cumsum()
    f["plan_cumulative"] = f.groupby(list(config.segments)).plan.cumsum()
    f["fact_cumulative"] = f.groupby(list(config.segments)).fact.transform(lambda s: s.cumsum(skipna=False))
    f["signed_error_daily"] = f.forecast_daily - f.fact
    f["signed_error_cumulative"] = f.forecast_cumulative.fillna(0) - f.fact_cumulative
    total = f.groupby("date", as_index=False).agg(
        plan_daily=("plan", "sum"), forecast_daily=("forecast_daily", "sum"),
        observed_fact_daily=("fact", lambda s: s.sum(min_count=1)),
        missing_actual_cells=("missing_actual", "sum"), unplanned_actual_cells=("unplanned_actual", "sum"))
    total["fact_daily"] = total.observed_fact_daily.where(total.missing_actual_cells.eq(0))
    for c in ["plan", "forecast", "fact"]:
        total[f"{c}_cumulative"] = total[f"{c}_daily"].cumsum(skipna=False)
    denominator = total.plan_cumulative.where(total.plan_cumulative.gt(0))
    total["actual_efficiency_pct"] = 100 * total.fact_cumulative / denominator
    total["forecast_efficiency_pct"] = 100 * total.forecast_cumulative / denominator
    total["signed_error_pp"] = total.forecast_efficiency_pct - total.actual_efficiency_pct
    total["absolute_error_pp"] = total.signed_error_pp.abs()
    total["within_tolerance"] = total.absolute_error_pp.le(config.tolerance_pp) & total.absolute_error_pp.notna()
    total["signed_error_money"] = total.forecast_cumulative - total.fact_cumulative
    total["month"] = total.date.dt.strftime("%Y-%m")
    return f.drop(columns="_merge"), total


def metrics(total, config):
    eligible = total.plan_cumulative.gt(0)
    valid = eligible & total.absolute_error_pp.notna()
    full_grid = bool(len(total) and pd.DatetimeIndex(total.date).equals(days(total.date.iloc[0].strftime("%Y-%m"))))
    complete = bool(full_grid and total.missing_actual_cells.sum() == 0 and total.unplanned_actual_cells.sum() == 0
                    and valid.sum() == eligible.sum() and eligible.any())
    errors = total.loc[valid, "absolute_error_pp"]
    return {
        "max_error_pp": float(errors.max()) if len(errors) else None,
        "mae_pp": float(errors.mean()) if len(errors) else None,
        "p90_error_pp": float(errors.quantile(0.9)) if len(errors) else None,
        "bias_pp": float(total.loc[valid, "signed_error_pp"].mean()) if len(errors) else None,
        "end_error_pp": float(total.absolute_error_pp.iloc[-1]) if len(total) and pd.notna(total.absolute_error_pp.iloc[-1]) else None,
        "hit_rate": float(total.loc[eligible, "within_tolerance"].mean()) if eligible.any() else None,
        "eligible_days": int(eligible.sum()), "scored_days": int(valid.sum()),
        "missing_actual_cells": int(total.missing_actual_cells.sum()),
        "unplanned_actual_cells": int(total.unplanned_actual_cells.sum()),
        "complete": complete, "full_month_grid": full_grid,
        "goal_achieved": bool(complete and errors.le(config.tolerance_pp).all()),
    }


def json_write(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str, allow_nan=False), encoding="utf-8")


class ForecastRunner:
    def __init__(self, config, *, history_df=None, plan_df=None):
        config = config.resolved()
        config.validate()
        if (history_df is None) != (plan_df is None):
            raise ValueError("Supply both history_df and plan_df when using DataFrame inputs")
        self.config = config
        self._from_frames = history_df is not None
        self.inputs = Inputs(config, plan_df=plan_df)
        self.builder = FeatureBuilder(config, self.inputs)
        if self._from_frames:
            self.history = prepare_actual(history_df, config, before=config.forecast_month)
        else:
            self.history = load_raw(config, before=config.forecast_month)
        self.selection = None
        self.status = []
        self.fold_log = []
        self.development_curves = pd.DataFrame()
        self.control_curves = pd.DataFrame()
        self.report_verified = False

    def audit(self):
        c = self.config
        h = raw_asof(self.history, c, origin(c.forecast_month))
        daily = daily_actual(h, c)
        report = {
            "history_start": str(h.date.min().date()), "history_end": str(h.date.max().date()),
            "raw_rows": len(h), "negative_daily_fact_rows": int(h.fact.lt(0).sum()),
            "raw_plan_negative_rows": int(h.target.lt(0).sum()),
            "raw_vintage_mode": "timestamped" if c.raw_available_column else "retrospective_reconstructed",
            "input_mode": "dataframes" if self._from_frames else "sqlite",
            "plan_source": "df_plan" if self._from_frames else f"{c.plan_table or c.raw_table}.{c.plan_value_column}",
            "plan_is_forecast": None if self._from_frames else c.plan_is_forecast,
            "plan_vintage_mode": "timestamped" if c.plan_available_column else "snapshot_no_plan_vintages",
            "calendar_mode": "approved_overrides" if self.inputs.calendar is not None else "holidays_library_with_workday_transfers",
            "fact_semantics": c.fact_mode,
            "plan_semantics": c.plan_mode, "semantics_verified_against_report": self.report_verified,
            "available_target_plan_rows": len(self.inputs.plan(c.forecast_month)),
            "external_reference": bool(c.reference_path),
            "reference_note": "external current-method forecast" if c.reference_path else "normalized historical cumulative shape with endpoint preservation",
        }
        rows = []
        for m in sorted(h.month.unique()):
            part = daily[daily.date.dt.strftime("%Y-%m") == m]
            expected = days(m)
            rows.append({"month": m, "observed_days": part.date.nunique(), "expected_days": len(expected),
                         "fact_sum": float(part.fact.sum()), "raw_plan_sum": float(part.raw_plan.sum())})
        # Build target features now so missing frozen portfolios/plan keys fail before a long backtest.
        self.builder.build(self.history, c.forecast_month)
        return report, pd.DataFrame(rows), daily

    def _predict_candidates(self, month):
        result = candidates(self.history, month, self.config, self.builder)
        self.status.extend({**entry, "origin_month": str(month)} for entry in result[3])
        return result[:3]

    def verify_report(self, checks_path, tolerance_money=0.01):
        """Reconcile daily sums against exported, independently calculated report totals."""
        checks = pd.read_csv(checks_path)
        if not {"date", "fact_daily", "plan_daily"} <= set(checks):
            raise ValueError("Report checks need date,fact_daily,plan_daily")
        if checks.empty:
            raise ValueError("Reference report checks are empty")
        numeric(checks, ["fact_daily", "plan_daily"])
        checks["date"] = pd.to_datetime(checks.date).dt.normalize()
        if checks.date.duplicated().any() or (checks.date >= month_start(self.config.forecast_month)).any():
            raise ValueError("Report check dates must be unique and before the target month")
        h = raw_asof(self.history, self.config, origin(self.config.forecast_month))
        computed = h.groupby("date", as_index=False).agg(computed_fact=("fact", "sum"), computed_plan=("target", "sum"))
        merged = checks.merge(computed, on="date", how="left", validate="one_to_one")
        merged["fact_difference"] = merged.computed_fact - merged.fact_daily
        merged["plan_difference"] = merged.computed_plan - merged.plan_daily
        if merged[["fact_difference", "plan_difference"]].isna().any().any() or not (
                merged[["fact_difference", "plan_difference"]].abs() <= tolerance_money).all().all():
            raise ValueError("Daily amounts disagree with the reference report; inspect fact/plan semantics before fitting")
        self.report_verified = True
        return merged

    def _truth(self, month):
        # Evaluation truth may be finalized/revised after a prediction origin.
        # It is never passed back into that origin's features or fitted model.
        h = raw_asof(self.history, self.config, pd.Timestamp.max - pd.Timedelta(days=1))
        return daily_actual(h[h.month == str(month)], self.config)

    def _apply(self, x, leaves, totals, spec):
        if spec["leaf"] not in leaves or (spec["total"] is not None and spec["total"] not in totals):
            raise ValueError(f"Selected candidate is unavailable: {spec_id(spec)}")
        f = reconcile(x, leaves[spec["leaf"]], totals.get(spec["total"]), spec["weight"], self.config)
        f["model"] = spec_id(spec)
        return f

    def select(self):
        c = self.config
        rows, curves, specifications = [], [], {}
        expected_months = [str(m) for m in pd.period_range(c.development_start, c.development_end, freq="M")]
        for month in expected_months:
            try:
                x, leaves, totals = self._predict_candidates(month)
                truth = self._truth(month)
                month_complete = True
                for spec in specs(leaves, totals, c.blend_weights):
                    key = spec_id(spec)
                    pred = self._apply(x, leaves, totals, spec)
                    _, total = evaluate(pred, truth, c)
                    score = metrics(total, c)
                    if not score["complete"]:
                        month_complete = False
                        break
                    specifications[key] = spec
                    rows.append({"candidate": key, "month": month, **score})
                    total["candidate"] = key
                    curves.append(total)
                if not month_complete:
                    rows = [r for r in rows if r["month"] != month]
                    curves = [r for r in curves if r.month.iloc[0] != month]
                    raise ValueError("incomplete or unplanned facts on evaluation grid")
                self.fold_log.append({"month": month, "phase": "development", "status": "ok"})
                print(f"Development {month}: {len(leaves)} segment models, {len(totals)} total models")
            except ValueError as exc:
                self.fold_log.append({"month": month, "phase": "development", "status": "failed", "reason": str(exc)})
                if c.require_all_folds:
                    raise ValueError(f"Development fold {month} failed: {exc}") from exc
                print(f"Development {month}: excluded: {exc}")
        scores = pd.DataFrame(rows)
        if scores.empty:
            raise ValueError("No complete development folds; no model has been selected")
        complete_months = scores.month.nunique()
        self.development_curves = pd.concat(curves, ignore_index=True)
        ranking = self.development_curves.groupby("candidate", as_index=False).agg(
            max_error_pp=("absolute_error_pp", "max"), mae_pp=("absolute_error_pp", "mean"),
            months=("month", "nunique"), hit_rate=("within_tolerance", "mean"))
        # Candidates must compete on exactly the same available months.
        ranking = ranking[ranking.months.eq(complete_months)].sort_values(
            ["max_error_pp", "mae_pp", "candidate"], kind="stable").reset_index(drop=True)
        if ranking.empty:
            raise ValueError("No candidate was available on every development fold")
        chosen = ranking.candidate.iloc[0]
        self.selection = specifications[chosen]
        self.development_scores, self.ranking = scores, ranking
        return ranking, dict(self.selection)

    def control(self):
        if self.selection is None:
            raise ValueError("Run select() first, or load a previously frozen selection")
        c = self.config
        rows, curves = [], []
        for m in pd.period_range(c.control_start, c.control_end, freq="M"):
            month = str(m)
            x, leaves, totals = self._predict_candidates(month)
            pred = self._apply(x, leaves, totals, self.selection)
            _, total = evaluate(pred, self._truth(month), c)
            score = metrics(total, c)
            if not score["complete"] and c.require_all_folds:
                raise ValueError(f"Incomplete control fold {month}")
            rows.append({"month": month, "candidate": spec_id(self.selection), **score})
            curves.append(total)
            self.fold_log.append({"month": month, "phase": "control", "status": "ok" if score["complete"] else "incomplete"})
            print(f"Control {month}: max error {score['max_error_pp']} pp")
        self.control_curves = pd.concat(curves, ignore_index=True)
        self.control_scores = pd.DataFrame(rows)
        return self.control_scores

    def forecast(self, selection=None, save=True):
        """Returns a frozen forecast without reading a single target-month actual."""
        if selection is not None:
            self.selection = dict(selection)
        if self.selection is None:
            raise ValueError("Run select() or supply a frozen selection")
        x, leaves, totals = self._predict_candidates(self.config.forecast_month)
        pred = self._apply(x, leaves, totals, self.selection)
        pred["forecast_efficiency_pct"] = 100 * pred.forecast_cumulative / pred.plan_cumulative.where(pred.plan_cumulative.gt(0))
        self.forecast_frame = pred
        if save:
            self.save_forecast(pred)
        return pred

    def save_forecast(self, pred):
        c = self.config
        out = Path(c.output_dir)
        out.mkdir(parents=True, exist_ok=True)
        payload = pred.to_csv(index=False, lineterminator="\n").encode("utf-8-sig")
        forecast_path = out / "forecast.csv"
        if forecast_path.exists() and forecast_path.read_bytes() != payload:
            raise FileExistsError(f"A different frozen forecast already exists in {out}; choose a new output_dir")
        forecast_path.write_bytes(payload)
        total = pred.groupby("date", as_index=False).agg(
            plan_daily=("plan", "sum"), forecast_daily=("forecast_daily", "sum"))
        total["plan_cumulative"] = total.plan_daily.cumsum()
        total["forecast_cumulative"] = total.forecast_daily.cumsum()
        total["forecast_efficiency_pct"] = 100 * total.forecast_cumulative / total.plan_cumulative.where(total.plan_cumulative.gt(0))
        total["origin"] = origin(c.forecast_month)
        total.to_csv(out / "forecast_total.csv", index=False, encoding="utf-8-sig")
        save_forecast_chart(total, c, out / "forecast.html")
        for attr, name in [("ranking", "development_ranking"), ("development_scores", "development_scores"),
                           ("control_scores", "control_scores"), ("control_curves", "control_curves"),
                           ("development_curves", "development_curves")]:
            frame = getattr(self, attr, None)
            if frame is not None and not frame.empty:
                frame.to_csv(out / f"{name}.csv", index=False, encoding="utf-8-sig")
        pd.DataFrame(self.status).to_csv(out / "model_status.csv", index=False, encoding="utf-8-sig")
        pd.DataFrame(self.fold_log).to_csv(out / "fold_log.csv", index=False, encoding="utf-8-sig")
        history = raw_asof(self.history, c, origin(c.forecast_month))
        digest = hashlib.sha256(pd.util.hash_pandas_object(history, index=False).to_numpy().tobytes()).hexdigest()
        versions = {}
        for package in ["pandas", "numpy", "catboost", "statsmodels", "holidays", "plotly"]:
            try:
                versions[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                versions[package] = None
        inputs = {}
        for m in sorted(set(history.month.unique()) | {c.forecast_month}):
            try:
                frame = self.inputs.plan(m)
                inputs[m] = hashlib.sha256(pd.util.hash_pandas_object(frame, index=False).to_numpy().tobytes()).hexdigest()
            except ValueError:
                continue
        audit, monthly, daily = self.audit()
        json_write(out / "audit.json", audit)
        monthly.to_csv(out / "history_months.csv", index=False, encoding="utf-8-sig")
        daily.to_csv(out / "history_daily_audit.csv", index=False, encoding="utf-8-sig")
        # Hash prediction inputs only, never the database file containing target actuals.
        json_write(out / "manifest.json", {"config": c.as_dict(), "selection": self.selection,
                   "history_sha256": digest, "plan_sha256": inputs, "versions": versions,
                   "forecast_sha256": hashlib.sha256((out / "forecast.csv").read_bytes()).hexdigest(),
                   "input_features_sha256": hashlib.sha256(pd.util.hash_pandas_object(
                       self.builder.build(self.history, c.forecast_month), index=False).to_numpy().tobytes()).hexdigest(),
                   "source_mode": audit["raw_vintage_mode"],
                   "input_mode": audit["input_mode"],
                   "asof_guarantee": "user-provided availability metadata; historical revisions may be unobservable without raw vintages"})

    def compare(self, forecast=None, save=True, *, actual_df=None):
        pred = self.forecast_frame if forecast is None else forecast
        if actual_df is not None:
            actual_raw = prepare_actual(actual_df, self.config, month=self.config.forecast_month)
        elif self._from_frames:
            raise ValueError("Supply actual_df to compare a DataFrame forecast with actuals")
        else:
            actual_raw = load_raw(self.config, month=self.config.forecast_month)
        # Evaluation uses the latest available actual revision, not a fictitious pre-month snapshot.
        actual_raw = raw_asof(actual_raw, self.config, pd.Timestamp.max - pd.Timedelta(days=1))
        segment, total = evaluate(pred, daily_actual(actual_raw, self.config), self.config)
        score = metrics(total, self.config)
        if save:
            save_report(segment, total, score, self.config)
        return segment, total, score


def run(df, plan_df=None, actual_df=None, *, config=None, forecast_month=None,
        fact_mode=None, plan_mode=None, save=True, report_checks_path=None):
    """Forecast from a date/fact/plan frame or from separate RR frames."""
    config = Config() if config is None else config
    overrides = {key: value for key, value in {
        "forecast_month": forecast_month, "fact_mode": fact_mode, "plan_mode": plan_mode
    }.items() if value is not None}
    config = replace(config, **overrides).resolved()
    config.validate()
    if plan_df is None:
        config = replace(config, plan_value_column="target")
        history_df, plan_df, combined_actual = split_dated_frame(df, config)
        if actual_df is None:
            actual_df = combined_actual
    else:
        history_df = df
    runner = ForecastRunner(config, history_df=history_df, plan_df=plan_df)
    audit, history_months, _ = runner.audit()
    report_checks = runner.verify_report(report_checks_path) if report_checks_path else None
    if report_checks is not None:
        audit["semantics_verified_against_report"] = True
    ranking, selection = runner.select()
    control = runner.control()
    forecast = runner.forecast(save=save)

    segment_comparison = comparison = score = None
    if actual_df is not None and not actual_df.empty:
        segment_comparison, comparison, score = runner.compare(actual_df=actual_df, save=save)
    return {
        "forecast": forecast, "comparison": comparison, "segment_comparison": segment_comparison,
        "metrics": score, "ranking": ranking, "control": control, "selection": selection,
        "audit": audit, "history_months": history_months, "report_checks": report_checks,
    }


## reporting

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def save_forecast_chart(total, config, path):
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=("Прогноз накопленного выполнения, %", "Накопленные суммы, руб."))
    fig.add_trace(go.Scatter(x=total.date, y=total.forecast_efficiency_pct, name="Прогноз выполнения"), row=1, col=1)
    for column, name in [("forecast_cumulative", "Прогноз сбора"), ("plan_cumulative", "Утверждённый план")]:
        fig.add_trace(go.Scatter(x=total.date, y=total[column], name=name), row=2, col=1)
    fig.update_layout(title=f"{config.forecast_month}: прогноз зафиксирован до начала месяца",
                      template="plotly_white", height=720, hovermode="x unified")
    fig.write_html(path, include_plotlyjs=True, full_html=True)


def grouped_breakdown(segment, dimensions):
    def strict_sum(s):
        return s.sum(min_count=len(s))
    return segment.groupby([*dimensions, "date"], as_index=False).agg(
        plan_daily=("plan", "sum"), forecast_daily=("forecast_daily", "sum"),
        fact_daily=("fact", strict_sum), forecast_cumulative=("forecast_cumulative", "sum"),
        fact_cumulative=("fact_cumulative", strict_sum),
        signed_error_cumulative=("signed_error_cumulative", strict_sum),
        reconciliation_cumulative=("reconciliation_cumulative", "sum"))


def save_report(segment, total, score, config):
    out = Path(config.output_dir)
    out.mkdir(parents=True, exist_ok=True)
    segment.to_csv(out / "comparison_segments.csv", index=False, encoding="utf-8-sig")
    total.to_csv(out / "comparison_total.csv", index=False, encoding="utf-8-sig")
    json_write(out / "metrics.json", score)
    for dims in [["subproduct"], ["dpd_bucket"], ["pvr_bucket"], ["subproduct", "dpd_bucket"]]:
        if set(dims) <= set(segment):
            grouped_breakdown(segment, dims).to_csv(out / ("by_" + "_".join(dims) + ".csv"),
                                                  index=False, encoding="utf-8-sig")
    categories = total[["date", "absolute_error_pp"]].copy()
    calendar = FeatureBuilder(config, Inputs(config)).calendar(config.forecast_month)
    categories = categories.merge(calendar, on="date")
    category_rows = []
    for name, mask in [("first_5_days", categories.day.le(5)),
                       ("weekends", categories.is_weekend.eq(1)),
                       ("holidays", categories.is_holiday.eq(1)),
                       ("ordinary_workdays", categories.is_workday.eq(1) & categories.is_holiday.eq(0))]:
        errors = categories.loc[mask, "absolute_error_pp"].dropna()
        category_rows.append({"category": name, "scored_days": len(errors),
                              "max_error_pp": errors.max(), "mae_pp": errors.mean()})
    pd.DataFrame(category_rows).to_csv(out / "calendar_error_groups.csv", index=False, encoding="utf-8-sig")
    fig = comparison_figure(total, config)
    fig.write_html(out / "comparison.html", include_plotlyjs=True, full_html=True)
    save_static_chart(total, config, out / "comparison.png")
    if score["complete"]:
        outcome = "ЦЕЛЬ ДОСТИГНУТА" if score["goal_achieved"] else "ЦЕЛЬ НЕ ДОСТИГНУТА"
    else:
        outcome = "ОЦЕНКА НЕПОЛНАЯ: пропущенные факты или сегменты вне исходного плана"
    failed = total[total.absolute_error_pp.gt(config.tolerance_pp)]
    text = (f"# {config.forecast_month}: {outcome}\n\n"
            f"Допуск: {config.tolerance_pp} п.п. на каждую дату с положительным планом.\n\n"
            f"Максимальная ошибка: {score['max_error_pp']} п.п.\n\n"
            f"Средняя ошибка: {score['mae_pp']} п.п.\n\n"
            f"Доля попаданий: {score['hit_rate']}\n\n"
            f"Даты превышения допуска: {', '.join(failed.date.dt.strftime('%Y-%m-%d')) or 'нет среди оценённых дат'}.\n\n"
            "Денежный вклад сегментов в отклонение и поправки согласования сохранены в comparison_segments.csv. "
            "Календарные группы и сегментные вклады описывают отклонения, но сами по себе не доказывают их причины.\n")
    (out / "report.md").write_text(text, encoding="utf-8")
    return fig


def comparison_figure(total, config):
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                        subplot_titles=("Накопленное выполнение плана, %", "Накопленный сбор и план, руб.",
                                        "Сбор за день, руб.", "Абсолютная ошибка эффективности, п.п."))
    x = total.date
    actual = total.actual_efficiency_pct
    fig.add_trace(go.Scatter(x=x, y=actual - config.tolerance_pp, line_width=0,
                            name="Нижняя граница", showlegend=False, connectgaps=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=x, y=actual + config.tolerance_pp, line_width=0, fill="tonexty",
                            fillcolor="rgba(35,150,100,0.14)", name=f"Допуск ±{config.tolerance_pp} п.п.", connectgaps=False), row=1, col=1)
    for col, label, color in [("actual_efficiency_pct", "Факт, %", "#218c65"),
                              ("forecast_efficiency_pct", "Прогноз, %", "#356ad2")]:
        fig.add_trace(go.Scatter(x=x, y=total[col], name=label, line_color=color, connectgaps=False), row=1, col=1)
    for col, label in [("fact_cumulative", "Факт накопленный"), ("forecast_cumulative", "Прогноз накопленный"),
                       ("plan_cumulative", "План накопленный")]:
        fig.add_trace(go.Scatter(x=x, y=total[col], name=label, connectgaps=False), row=2, col=1)
    for col, label in [("fact_daily", "Факт за день"), ("forecast_daily", "Прогноз за день")]:
        fig.add_trace(go.Scatter(x=x, y=total[col], name=label, connectgaps=False), row=3, col=1)
    fig.add_trace(go.Bar(x=x, y=total.absolute_error_pp, name="Ошибка, п.п.",
                        marker_color=np.where(total.within_tolerance, "#218c65", "#cb4747")), row=4, col=1)
    fig.add_hline(y=config.tolerance_pp, line_dash="dash", line_color="#cb4747", row=4, col=1)
    fig.update_layout(title=f"Прогноз и факт · {config.forecast_month} · старт до начала месяца",
                      height=1150, template="plotly_white", hovermode="x unified")
    return fig


def save_static_chart(total, config, path):
    import os
    cache = Path(config.output_dir) / ".matplotlib"
    cache.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("MPLCONFIGDIR", str(cache.resolve()))
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(4, 1, figsize=(13, 14), sharex=True)
    x = total.date
    axes[0].plot(x, total.actual_efficiency_pct, label="Факт", color="#218c65")
    axes[0].plot(x, total.forecast_efficiency_pct, label="Прогноз", color="#356ad2")
    axes[0].fill_between(x, total.actual_efficiency_pct - config.tolerance_pp,
                         total.actual_efficiency_pct + config.tolerance_pp, alpha=0.15, color="#218c65")
    axes[0].set_ylabel("Выполнение, %")
    for c, label in [("fact_cumulative", "Факт"), ("forecast_cumulative", "Прогноз"), ("plan_cumulative", "План")]:
        axes[1].plot(x, total[c], label=label)
    axes[1].set_ylabel("Накопленно, руб.")
    axes[2].plot(x, total.fact_daily, label="Факт")
    axes[2].plot(x, total.forecast_daily, label="Прогноз")
    axes[2].set_ylabel("За день, руб.")
    axes[3].bar(x, total.absolute_error_pp, color=np.where(total.within_tolerance, "#218c65", "#cb4747"))
    axes[3].axhline(config.tolerance_pp, ls="--", color="#cb4747", label="Допуск")
    axes[3].set_ylabel("Ошибка, п.п.")
    for ax in axes:
        ax.legend(loc="upper left")
        ax.grid(alpha=0.15)
    axes[-1].tick_params(axis="x", rotation=35)
    fig.suptitle(f"{config.forecast_month}: прогноз из конца предыдущего месяца")
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


## demo

In [ ]:
"""Deterministic artificial fixture. Never use it as evidence of business accuracy."""
from dataclasses import replace
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd


def demo_rows():
    raw = []
    for month in pd.period_range("2026-01", "2026-10", freq="M"):
        for product, base in [("ПК", 10000.0), ("КК", 6500.0)]:
            dims = dict(subproduct=product, dpd_bucket="1. 1-30", pvr_bucket="1. [1-15]", terbank="МБ", inflow="0")
            for date in days(str(month)):
                plan = base * (0.45 if date.day_of_week >= 5 else 1.0) * (1.8 if date.day <= 3 else 1.0)
                fact = plan * (0.95 + 0.006 * np.sin(date.day / 4) + 0.002 * month.month)
                if date.day == 19 and product == "КК":
                    fact = -30.0  # A valid signed adjustment, not a missing value.
                raw.append(dict(report_month=str(month.start_time.date()), report_dt=str(date.date()),
                                dt_in=str(month.start_time.date()), **dims, w1=1_000_000.0,
                                w2=1_000_000.0-fact, cession=0.0, write_off=0.0, target=plan,
                                acc_count=100, avg_dpd=15.0, avg_pvr=10.0, is_forecast=0,
                                plan_known_at=str(origin(str(month)).date())))
    return pd.DataFrame(raw)


def make_demo(directory="outputs/demo_inputs"):
    out = Path(directory)
    out.mkdir(parents=True, exist_ok=True)
    database = out / "artificial.db"
    with sqlite3.connect(database) as conn:
        demo_rows().to_sql("rr1plus_data_raw", conn, index=False, if_exists="replace")
    conn.close()
    return Config(db_path=str(database), plan_available_column="plan_known_at", forecast_month="2026-10",
                  fact_mode="daily", plan_mode="daily",
                  output_dir=str(out.parent / "demo_results"), min_train_months=2,
                  development_start="2026-04", development_end="2026-06", control_start="2026-07", control_end="2026-09",
                  catboost_depths=(4,), catboost_iterations=60)


## Настройки и запуск

In [ ]:
DEMO = False

if DEMO:
    config = replace(make_demo("outputs/notebook_demo_inputs"),
                     development_start="2026-04", development_end="2026-04",
                     control_start="2026-09", control_end="2026-09",
                     catboost_iterations=20)
    raw = demo_rows()
    raw["fact"] = raw.w1 - raw.w2 - raw.cession - raw.write_off
    df = raw.rename(columns={"report_dt": "date", "target": "plan"})
else:
    config = Config(
        forecast_month="2026-11",  # измените целевой месяц
        fact_mode="daily",         # либо cumulative после сверки с отчётом
        plan_mode="daily",         # выбирается отдельно от факта
        output_dir="outputs/forecast",
    )
    # df = ваш_коннектор(...)  # date, fact, plan; включая известный будущий план
    if "df" not in globals():
        raise ValueError("Передайте DataFrame df или включите DEMO = True")

result = run(df, config=config)
display(result["ranking"].head(10))
display(result["control"])
display(result["forecast"].head())


In [ ]:
if result["comparison"] is not None:
    display(result["metrics"])
    comparison_figure(result["comparison"], config).show()
else:
    print("Прогноз сохранён. Для оценки передайте факты месяца отдельным этапом.")
